# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and process the FAIR\u2072 dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets, their fields and columns with @id
import json
recset_ids = []
print('Record Sets Found:')
for recset in dataset.record_sets:
    print(f'  RecordSet name: {recset.name}  |  @id: {recset.id}')
    recset_ids.append(recset.id)
    print('    Fields:')
    for field in recset.fields:
        print(f"      Field name: {field.name}  |  @id: {field.id}  |  Data type: {field.data_type}")
    print('    Columns:')
    for col in recset.columns:
        print(f"      Column name: {col.name}  |  @id: {col.id}  |  Data type: {col.data_type}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For demonstration: pick the first record set and load its data
# You may replace the @id below with another @id from the overview to explore other sets
assert len(recset_ids) > 0, "No record sets found!"
record_sets = recset_ids
dataframes = {}

for rs_id in record_sets:
    print(f"Loading records for: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns in {rs_id}: {df.columns.tolist()}")
    else:
        print(f"No records found for {rs_id}")
# Preview data from the first populated record set
for rs_id, df in dataframes.items():
    print(f"\nData sample from {rs_id}")
    display(df.head())
    break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.


In [ ]:
# Example: Filter, normalize, and group on a numeric column.
# Select the first populated DataFrame
if dataframes:
    rs_id, df = next(iter(dataframes.items()))
    # Attempt to select a likely numeric field using known fields from proteomics datasets
    # For real usage, review the previous overview to pick the target column
    numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float, 'int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try string columns that could be converted
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                numeric_candidates.append(col)
            except Exception:
                continue
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Pick the first
        print(f"Analyzing field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric column if available
        non_num_cols = [c for c in df.columns if c != numeric_field and not pd.api.types.is_numeric_dtype(df[c])]
        if non_num_cols:
            group_field = non_num_cols[0]
            try:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(f"Grouped data by {group_field} (mean of {numeric_field}):")
                display(grouped_df.head())
            except Exception:
                print(f"Unable to group by {group_field} due to grouping error.")
        else:
            print("No suitable field found for grouping.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping field available, show boxplot
    if non_num_cols:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=non_num_cols[0], y=numeric_field)
        plt.title(f"{numeric_field} grouped by {non_num_cols[0]}")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook has demonstrated step-by-step loading and exploration of a dataset described by a Croissant schema using `mlcroissant`. We extracted metadata, enumerated record sets and fields by their `@id`, loaded data as DataFrames, performed basic EDA and visualizations, referencing all entities by their `@id`. For more advanced analyses, further domain knowledge and schema documentation is recommended.
